# P00b — DOSE Subnational GDP

Load and explore the DOSE (Database of Subnational Economic output). Convert to Arrow for downstream use.

**Source:** [DOSE V2.11](https://zenodo.org/records/16313760) — 46,851 region-year rows, 83 countries, 1,661 regions, 1953–2020. Includes sectoral breakdown (agriculture, manufacturing, services) and climate variables (temperature, precipitation).

In [1]:
include("phase00b/functions/load_phase00b.jl")

## 1. Load Subnational Panel

In [2]:
sub = dose_load_subnational();

In [3]:
describe(sub)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,Type
1,iso3,,ALB,,ZAF,0,String3
2,gid_1,,ALB.10_1,,ZAF.9_1,39,"Union{Missing, String15}"
3,country,,Albania,,Vietnam,0,String31
4,region,,ARMM,,Žilina,0,String
5,year,2000.03,1953,2003.0,2020,0,Int64
6,pop,4.40818e6,1811.0,1.0943e6,2.27943e8,2302,"Union{Missing, Float64}"
7,gdp_pc_usd2015,12872.5,2.57825,4905.85,1.98187e5,14,"Union{Missing, Float64}"
8,gdp_pc_ag_usd2015,659.134,0.0,463.65,23744.0,11537,"Union{Missing, Float64}"
9,gdp_pc_man_usd2015,4884.18,6.20136,2409.61,2.43331e5,11511,"Union{Missing, Float64}"


In [4]:
first(sub, 5)

Row,iso3,gid_1,country,region,year,pop,gdp_pc_usd2015,gdp_pc_ag_usd2015,gdp_pc_man_usd2015,gdp_pc_serv_usd2015,temp_annual,precip_annual
,String3,String15?,String31,String,Int64,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?
1,ALB,ALB.1_1,Albania,Berat,2010,151375.0,3419.07,missing,missing,missing,13.1238,1573.12
2,ALB,ALB.1_1,Albania,Berat,2011,148160.0,3752.33,missing,missing,missing,12.882,796.288
3,ALB,ALB.1_1,Albania,Berat,2012,145931.0,3248.64,missing,missing,missing,13.3735,1177.16
4,ALB,ALB.1_1,Albania,Berat,2013,145132.0,3363.27,missing,missing,missing,13.5354,1086.02
5,ALB,ALB.1_1,Albania,Berat,2014,143846.0,3456.43,missing,missing,missing,13.358,1198.51


## 2. Coverage

In [5]:
println("Rows:      $(nrow(sub))")
println("Years:     $(minimum(sub.year)) – $(maximum(sub.year))")
println("Countries: $(length(unique(sub.iso3)))")
println("Regions:   $(length(unique(sub.gid_1)))")
println()
# Regions per country
reg_per_country = combine(groupby(sub, :iso3), :gid_1 => (x -> length(unique(x))) => :n_regions)
sort!(reg_per_country, :n_regions, rev=true)
println("Top 10 by region count:")
first(reg_per_country, 10)

Rows:      46851
Years:     1953 – 2020
Countries: 83
Regions:   1662

Top 10 by region count:


Row,iso3,n_regions
,String3,Int64
1,TUR,81
2,RUS,79
3,THA,77
4,VNM,63
5,USA,51
6,KEN,48
7,JPN,47
8,ROU,42
9,IND,33


In [6]:
# Temporal coverage — years per country
yr_per_country = combine(groupby(sub, :iso3), :year => minimum => :first_year, :year => maximum => :last_year,
                         :year => length => :n_obs)
sort!(yr_per_country, :first_year)
println("Earliest start: $(minimum(yr_per_country.first_year))")
println("Latest end:     $(maximum(yr_per_country.last_year))")
println()
# Countries starting before 1960
println("Countries with data before 1960: $(count(yr_per_country.first_year .< 1960))")
println("Countries with data from 1990+:  $(count(yr_per_country.first_year .>= 1990))")

Earliest start: 1953
Latest end:     2020

Countries with data before 1960: 1
Countries with data from 1990+:  52


## 3. Missingness — Sectoral and Climate

In [7]:
# Missingness by era — which columns are worst, and is it temporal?
pre = filter(r -> r.year < 1990, sub)
post = filter(r -> r.year >= 1990, sub)

data_cols = [:pop, :gdp_pc_usd2015, :gdp_pc_ag_usd2015, :gdp_pc_man_usd2015, 
             :gdp_pc_serv_usd2015, :temp_annual, :precip_annual]

println("Column                    Pre-1990 (n=$(nrow(pre)))    Post-1990 (n=$(nrow(post)))    Overall (n=$(nrow(sub)))")
println("─"^90)
for col in data_cols
    pre_miss = round(100 * count(ismissing, pre[!, col]) / nrow(pre), digits=1)
    post_miss = round(100 * count(ismissing, post[!, col]) / nrow(post), digits=1)
    all_miss = round(100 * count(ismissing, sub[!, col]) / nrow(sub), digits=1)
    println("  $(rpad(string(col), 25)) $(lpad(string(pre_miss), 5))% missing      $(lpad(string(post_miss), 5))% missing      $(lpad(string(all_miss), 5))% missing")
end

Column                    Pre-1990 (n=10426)    Post-1990 (n=36425)    Overall (n=46851)
──────────────────────────────────────────────────────────────────────────────────────────
  pop                         9.7% missing        3.5% missing        4.9% missing
  gdp_pc_usd2015              0.1% missing        0.0% missing        0.0% missing
  gdp_pc_ag_usd2015          37.5% missing       20.9% missing       24.6% missing
  gdp_pc_man_usd2015         37.5% missing       20.9% missing       24.6% missing
  gdp_pc_serv_usd2015        37.6% missing       20.9% missing       24.6% missing
  temp_annual                49.7% missing        6.1% missing       15.8% missing
  precip_annual              49.7% missing        6.1% missing       15.8% missing


## 3b. Country Code Alignment with QoG

In [8]:
using Arrow

# Load augmented QoG country codes (Phase 0 output)
qog = DataFrame(Arrow.Table("data/qog_std_ts_jan25_aug.arrow"))
qog_codes = Set(unique(skipmissing(qog.ident_ccodealp)))
dose_codes = Set(unique(sub.iso3))

in_dose_not_qog = sort(collect(setdiff(dose_codes, qog_codes)))
in_qog_not_dose = sort(collect(setdiff(qog_codes, dose_codes)))
matched = length(intersect(dose_codes, qog_codes))

println("QoG countries:  $(length(qog_codes))")
println("DOSE countries: $(length(dose_codes))")
println("Matched:        $matched")
println()
println("In DOSE but NOT in QoG ($(length(in_dose_not_qog))):")
for c in in_dose_not_qog
    name = first(filter(r -> r.iso3 == String3(c), sub)).country
    println("  $c — $name")
end
println()
println("In QoG but NOT in DOSE ($(length(in_qog_not_dose))):")
for c in in_qog_not_dose
    println("  $c")
end

QoG countries:  202
DOSE countries: 83
Matched:        82

In DOSE but NOT in QoG (1):
  ANT — Netherlands Antilles

In QoG but NOT in DOSE (120):
  AFG
  AGO
  AND
  ARM
  ATG
  BDI
  BEN
  BFA
  BGD
  BHR
  BLZ
  BRB
  BRN
  BTN
  BWA
  CAF
  CIV
  CMR
  COD
  COG
  COM
  CPV
  CRI
  CSK
  CUB
  CYP
  DDR
  DJI
  DMA
  DOM
  DZA
  ERI
  FJI
  FSM
  GAB
  GHA
  GIN
  GMB
  GNB
  GNQ
  GRD
  GUY
  HTI
  IRQ
  ISL
  ISR
  JAM
  JOR
  KHM
  KIR
  KNA
  KWT
  LBN
  LBR
  LBY
  LCA
  LIE
  LSO
  LUX
  MCO
  MDA
  MDG
  MDV
  MHL
  MLI
  MLT
  MMR
  MNE
  MRT
  MUS
  MWI
  NAM
  NER
  NIC
  NRU
  OMN
  PLW
  PNG
  PRK
  QAT
  RWA
  SAU
  SCG
  SDN
  SEN
  SGP
  SLB
  SLE
  SLV
  SMR
  SOM
  SSD
  STP
  SUN
  SUR
  SWZ
  SYC
  SYR
  TCD
  TGO
  TJK
  TKM
  TLS
  TON
  TTO
  TUN
  TUV
  TWN
  UGA
  VCT
  VDR
  VEN
  VUT
  WSM
  XTI
  YEM
  YMD
  YUG
  ZMB
  ZWE


## 4. Aggregate to Country-Year

In [9]:
using Statistics
nat = dose_aggregate_national(sub)
describe(nat)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,Type
1,iso3,,ALB,,ZAF,0,String3
2,country,,Albania,,Vietnam,0,String31
3,year,2001.64,1960,2005.0,2020,0,Int64
4,total_pop,9.38655e7,22304.0,2.73921e7,1.40977e9,0,Float64
5,n_regions,21.2882,1,16.0,81,0,Int64
6,gdp_pc_usd2015_wtd,15555.7,75.2573,7871.86,90655.7,0,Float64
7,gdp_pc_ag_usd2015_wtd,540.706,76.0361,471.739,2456.82,409,"Union{Missing, Float64}"
8,gdp_pc_man_usd2015_wtd,5035.48,30.2562,3520.6,30999.4,408,"Union{Missing, Float64}"
9,gdp_pc_serv_usd2015_wtd,11437.8,63.8251,6448.66,58997.6,409,"Union{Missing, Float64}"


In [10]:
# Spot check: compare a few countries
filter(r -> r.iso3 == "USA" && r.year == 2015, nat)

Row,iso3,country,year,total_pop,n_regions,gdp_pc_usd2015_wtd,gdp_pc_ag_usd2015_wtd,gdp_pc_man_usd2015_wtd,gdp_pc_serv_usd2015_wtd,temp_annual_mean,precip_annual_mean
,String3,String31,Int64,Float64,Int64,Float64,Float64?,Float64?,Float64?,Float64?,Float64?
1,USA,USA,2015,3.20635e8,51,56532.6,568.505,19069.4,36894.7,12.2667,998.611


## 5. Subnational Dispersion (Preview for Signatures 3 & 4)

Quick look at within-country GDP dispersion — the foundation for scale invariance and fractal structure testing.

In [11]:
# Within-country coefficient of variation of GDP per capita (2015)
yr2015 = filter(r -> r.year == 2015 && !ismissing(r.gdp_pc_usd2015), sub)
dispersion = combine(groupby(yr2015, [:iso3, :country]),
    :gdp_pc_usd2015 => std => :sd_gdp,
    :gdp_pc_usd2015 => mean => :mean_gdp,
    nrow => :n_regions
)
dispersion[!, :cv] = dispersion.sd_gdp ./ dispersion.mean_gdp
filter!(r -> r.n_regions >= 3, dispersion)  # need at least 3 regions for meaningful CV
sort!(dispersion, :cv)
println("Top 15 most internally equal (lowest CV of subnational GDP pc, 2015):")
first(select(dispersion, :iso3, :country, :n_regions, :mean_gdp, :cv), 15)

Top 15 most internally equal (lowest CV of subnational GDP pc, 2015):


Row,iso3,country,n_regions,mean_gdp,cv
,String3,String31,Int64,Float64,Float64
1,BHS,Bahamas,3,29778.1,0.0826592
2,IRL,Ireland,26,27276.4,0.114484
3,GBR,UK,4,34179.0,0.162432
4,BIH,Bosnia and Herzegovina,3,4443.86,0.162788
5,SWE,Sweden,21,44939.4,0.16774
6,PRT,Portugal,20,17391.1,0.182144
7,AUT,Austria,9,43753.9,0.183058
8,ANT,Netherlands Antilles,3,28388.3,0.189755
9,JPN,Japan,47,31994.7,0.198209


## 6. Write Arrow

In [13]:
# Arrow.write(PATH_DOSE_SUBNATIONAL_ARROW, sub)
# println("  → $(PATH_DOSE_SUBNATIONAL_ARROW)  ($(round(filesize(PATH_DOSE_SUBNATIONAL_ARROW) / 1024^2, digits=1)) MB)")

# Arrow.write(PATH_DOSE_NATIONAL_ARROW, nat)
# println("  → $(PATH_DOSE_NATIONAL_ARROW)  ($(round(filesize(PATH_DOSE_NATIONAL_ARROW) / 1024^2, digits=1)) MB)")